In [ ]:
import os
from os import path
import random
import pickle

import numpy as np
import networkx as nx

import matplotlib.pyplot as plt
import matplotlib.colors as clr

from utils.plotting import color_graph_by_attribute
from utils.assortativity import calc_weighted_attrib_assortativity, calc_weighted_attrib_assortativity_sig

In [ ]:
data_folder = './data'
output_folder = './output'
if not path.exists(output_folder):
    os.makedirs(output_folder)

In [ ]:
with open(path.join(data_folder, 'network-largest_conn_comp.pickle'), 'rb') as f:
    G = pickle.load(f)

with open(path.join(data_folder, 'network-node-positions.pickle'), 'rb') as f:
    Gpos = pickle.load(f)

with open(path.join(data_folder, 'users.pickle'), 'rb') as f:
    users = pickle.load(f)

with open(path.join(data_folder, 'poisson_model.pickle'), 'rb') as f:
    poisson_model_h2b = pickle.load(f)

## Plotting the network

In [ ]:
plt.figure(figsize=(15, 15))

weights = list(nx.get_edge_attributes(G, 'weight').values())
wmin = np.min(weights)/2
wmax = np.max(weights)
edge_weights = [pow((w - wmin)/(wmax-wmin), 0.35) for w in weights] 

nx.draw_networkx(G, Gpos, with_labels=False, node_size=10, width=edge_weights, alpha=1)
plt.tight_layout()
plt.savefig(path.join(output_folder, 'network.png'))
plt.show()

In [ ]:
# Plot network colored by infected/not-infected state

# Binary color map for healthy/infected final status:
# https://matplotlib.org/3.1.0/gallery/color/named_colors.html
health_color = {0: clr.to_hex("cornflowerblue"),  # Susceptible
                1: clr.to_hex("darkorange"),      # Infected (index case)
                2: clr.to_hex("darkorange"),      # Infected (from someone else)
                3: clr.to_hex("darkorange"),      # Dead 
                4: clr.to_hex("darkorange"),      # Recovered 
                5: clr.to_hex("darkorange")       # Vaccinated 
               } 

colors = [health_color[G.nodes[node]["final_health_state"]] for node in G.nodes()]

plt.figure(figsize=(15, 15))
nx.draw_networkx(G, Gpos, node_color=colors, with_labels=False, node_size=20, width=edge_weights, alpha=1)
#nx.draw(G, Gpos, ax=ax, node_color=colors, with_labels=False, node_size=7, width=edge_weights, alpha=1)

plt.tight_layout()
plt.savefig(path.join(output_folder, 'network-final-health-status.png'))
plt.show()

## Degree distribution

The distribution of the degrees of the nodes in a network is another basic property. The degree of a node) is simply how many edges it has with other nodes. It measures the level of connectivity of each node. In the next cell, we calculate the histogram of the node degree, so that in each bin we have how many nodes in the network have a degree within that bin:

In [ ]:
# Calculate degree distribution
degrees = [degree for node, degree in G.degree()]
degree_distribution = {}
for degree in degrees:
    if degree in degree_distribution:
        degree_distribution[degree] += 1
    else:
        degree_distribution[degree] = 1

# Plot the degree histogram
plt.figure(figsize=(6, 4))
plt.bar(degree_distribution.keys(), degree_distribution.values())

plt.title("Degree Histogram")
plt.ylabel("Number of nodes")
plt.xlabel("Degree")
plt.xticks(range(0, np.max(degrees), 5))
plt.tight_layout()
plt.savefig(path.join(output_folder, 'degree_hist.png'))
plt.show()

Examination of this plot suggests a power law for the [distribution of the node degree](https://en.wikipedia.org/wiki/Degree_distribution), which indicates that the contact network of participants of the epigame at AUIB is [scale-free](https://en.wikipedia.org/wiki/Scale-free_network). Scale-free networks do not have an "epidemic threshold" ([paper showing](https://arxiv.org/abs/cond-mat/0010317) this result), which means that no matter how much the contact rate between individuals is reduced, an epidemic will still take place:

In [ ]:
from matplotlib.ticker import FuncFormatter

x = np.log10(np.array(list(degree_distribution.keys())))
y = np.log10(np.array(list(degree_distribution.values())) / G.number_of_nodes())

# Perform linear regression and reate regression line
slope, intercept = np.polyfit(x, y, 1)
yreg = slope * x + intercept

def power_formatter_y(y, pos):
    return f'{10 ** y:.3f}'

def power_formatter_x(x, pos):
    return f'{round(10 ** x)}'

plt.figure(figsize=(6, 4))
plt.plot(x, yreg, color='coral', zorder=1)
plt.scatter(x, y, color='cornflowerblue', zorder=2)
plt.title("Degree Distribution (Log-Log)")
plt.ylabel("Probability")
plt.xlabel("Degree")
plt.gca().yaxis.set_major_formatter(FuncFormatter(power_formatter_y))
plt.gca().xaxis.set_major_formatter(FuncFormatter(power_formatter_x))
plt.tight_layout()
plt.savefig(path.join(output_folder, 'degree_dist_loglog.png'))
plt.show()

## Assortativity calculations

In [ ]:
attribs_to_add = [
    'quarantine', 'no_quarantine', 'group',
    'S1_Q1', 'S1_Q2', 'S1_Q3', 'S1_Q4', 'S1_Q5',
    'S2_Q1', 'S2_Q2', 'S2_Q3', 'S2_Q4',
    'S3_Q1', 'S3_Q2', 'S3_Q3', 'S3_Q4', 'S3_Q5'
]

# Filter and reindex by keeping only the rows corresponding to nodes in the graph.
# It automatically creates rows with NaN values for any node in G that is not found in the users dataframe.
node_attributes_df = users[attribs_to_add].reindex(G.nodes())

# Add to graph in one go:
# to_dict('index') converts the dataframe to: {node_id: {'col1': val, 'col2': val...}}
nx.set_node_attributes(G, node_attributes_df.to_dict('index'))

In [ ]:
# Adds the prediction from the Poisson model as another attribute in the graph

# Initialize the dictionary with NaN for all nodes in the graph
prediction_dict = {node: np.nan for node in G.nodes()}

# Identify nodes that are in the graph and have complete data by checking 
# intersection of indices and dropping rows with missing values in required columns
required_cols = ['S2_Q1', 'S2_Q2', 'S2_Q3', 'S2_Q4', 'S1_Q5', 'group']
valid_mask = users.index.isin(G.nodes())
df_clean = users.loc[valid_mask, required_cols].dropna()

if not df_clean.empty:
    # 3. Create a copy to apply preprocessing safely
    df_for_prediction = df_clean.copy()

    # 4. Re-apply the "Anchoring" (Minus 1) exactly as you did for training
    cols_to_anchor = ['S2_Q1', 'S2_Q2', 'S2_Q3', 'S2_Q4', 'S1_Q5']
    for col in cols_to_anchor:
        df_for_prediction[col] = df_for_prediction[col] - 1

    # 5. Generate Predictions
    preds = poisson_model_h2b.predict(df_for_prediction)

    # 6. Update the dictionary 
    # This only updates keys that exist in 'preds'; the rest stay NaN
    prediction_dict.update(preds.to_dict())

nx.set_node_attributes(G, prediction_dict, 'predicted_quarantine')

attribs_to_add.append('predicted_quarantine')

In [ ]:
# Assortativity calculation using the DescrStatsW from statsmodels.stats.weightstats
# and the p-value using bootstrapping
for attrib in attribs_to_add:
    rho = calc_weighted_attrib_assortativity(G, attrib)
    pval = calc_weighted_attrib_assortativity_sig(G, attrib, rho, 10000)
    print(f'Assortativity value for {attrib} = {rho:.2f} (p-value = {pval:.5f})')

In [ ]:
weights = list(nx.get_edge_attributes(G, 'weight').values())
wmin = np.min(weights)/2
wmax = np.max(weights)
edge_weights = [pow((w - wmin)/(wmax-wmin), 0.35) for w in weights] 

In [ ]:
wm_dict = nx.get_node_attributes(G, 'no_quarantine')
wm_attrib = [wm_dict[idx] for idx in wm_dict]
color_graph_by_attribute(G, Gpos, wm_attrib, 'Node Coloring by no quarantine', edge_weights, path.join(output_folder, 'network-node-coloring-by-no-quarantine.png'))

In [ ]:
wm_dict = nx.get_node_attributes(G, 'S1_Q1')
wm_attrib = [wm_dict[idx] for idx in wm_dict]
color_graph_by_attribute(G, Gpos, wm_attrib, 'Node Coloring by S1_Q1 (real-life susceptibility)', edge_weights, path.join(output_folder, 'network-node-coloring-by-S1_Q1.png'))

In [ ]:
wm_dict = nx.get_node_attributes(G, 'S1_Q5')
wm_attrib = [wm_dict[idx] for idx in wm_dict]
color_graph_by_attribute(G, Gpos, wm_attrib, 'Node Coloring by S1_Q5 (gender)', edge_weights, path.join(output_folder, 'network-node-coloring-by-S1_Q5.png'))

In [ ]:
wm_dict = nx.get_node_attributes(G, 'S2_Q2')
wm_attrib = [wm_dict[idx] for idx in wm_dict]
color_graph_by_attribute(G, Gpos, wm_attrib, 'Node Coloring by S2_Q2 (in-game severity)', edge_weights, path.join(output_folder, 'network-node-coloring-by-S2_Q2.png'))

In [ ]:
wm_dict = nx.get_node_attributes(G, 'S2_Q4')
wm_attrib = [wm_dict[idx] for idx in wm_dict]
color_graph_by_attribute(G, Gpos, wm_attrib, 'Node Coloring by S2_Q4 (in-game benefit)', edge_weights, path.join(output_folder, 'network-node-coloring-by-S2_Q4.png'))